In [ ]:
!pip install google-generativeai langchain langchain-core

In [1]:
import google.generativeai as genai
from getpass import getpass
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableSequence, RunnablePassthrough
from langchain.memory import ConversationBufferMemory

In [2]:
api_key = getpass("🔑 Enter your Gemini API key: ")
genai.configure(api_key=api_key)

# Create a custom LLM class that wraps Gemini for LangChain compatibility
from langchain_core.language_models.llms import LLM
from typing import Optional, List, Mapping, Any

class GeminiLLM(LLM):
    model_name: str = "gemini-2.5-flash"
    
    @property
    def _llm_type(self) -> str:
        return "gemini"
    
    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs) -> str:
        model = genai.GenerativeModel(self.model_name)
        response = model.generate_content(prompt)
        return response.text
    
    @property
    def _identifying_params(self) -> Mapping[str, Any]:
        return {"model_name": self.model_name}

model = GeminiLLM()

## Create a Memory Object

In [3]:
# Simple in-memory conversation
memory = ConversationBufferMemory(memory_key="chat_history")

## Define Prompt with Memory

In [4]:
# Prompt template that uses memory
prompt = ChatPromptTemplate.from_template(
    "The following is a conversation so far:\n{chat_history}\n"
    "User: {user_input}\nAI:"
)

## Run a Conversational Loop

In [7]:
# --- Memory ---
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=False)

# --- Prompt ---
prompt = ChatPromptTemplate.from_template(
    "The following is a conversation so far:\n{chat_history}\n"
    "User: {user_input}\nAI:"
)

# --- Chain ---
# Replaces your current "Chain" block
chain = (
    RunnablePassthrough()
    | prompt
    | (lambda x: x.to_string())
    | model
    | StrOutputParser()
)

# --- Multi-turn conversation ---
# First turn
memory_vars = memory.load_memory_variables({})
user_input1 = "Hello!"
input1 = {
    "user_input": user_input1,
    "chat_history": memory_vars["chat_history"]
}
response1 = chain.invoke(input1)
print("AI:", response1)

# Save to memory
memory.save_context({"user_input": user_input1}, {"output": response1})

# Second turn
memory_vars = memory.load_memory_variables({})
user_input2 = "How are you?"
input2 = {
    "user_input": user_input2,
    "chat_history": memory_vars["chat_history"]
}
response2 = chain.invoke(input2)
print("AI:", response2)

AI: Hello! How can I help you today?
AI: I'm doing well, thank you for asking! How are you doing today?
